## Background Research

**What is HotpotQA?**
HotpotQA is a QA dataset where most questions require reasoning across *two*
Wikipedia paragraphs, not one. Each example includes "supporting facts" —
sentence-level annotations showing exactly which sentences, in which
paragraphs, were needed to answer. This makes it a good testbed for multi-hop
reasoning, and a good stress-test for standard RAG, which is built around
single retrieve-then-generate.

**Why single-shot RAG struggles here**
A standard RAG pipeline embeds the question once, retrieves top-k passages
once, and generates an answer once. If the answer requires a fact from
document A ("Which team did X play for") to know what to search for in
document B ("What year did that team win the championship"), a single
retrieval pass over the *original* question will often miss document B
entirely — the question text never mentions the team name.

**The agentic alternative**
An agent with tool-use can decide, after seeing the first retrieval, that it
needs another search — using an intermediate fact it just learned as the next
query. This turns retrieval from a fixed one-shot step into a controlled loop:
retrieve → reason → decide (answer or retrieve again) → repeat.

**Plan for this project**
1. Build a baseline: single-retrieval RAG on HotpotQA (expected to do
   reasonably on single-hop-friendly questions, poorly on genuine multi-hop
   ones — this is the hypothesis to test, not an assumed result).
2. Build the agentic version: Claude with a `search_documents` tool it can
   call repeatedly, deciding for itself when it has enough evidence.
3. Score both with exact-match and F1 against HotpotQA's gold answers, on the
   same held-out slice, so the comparison is fair.
4. Read through the *actual* transcripts afterward and write up specific
   cases — including at least one clear failure — rather than only reporting
   the aggregate number.

Nothing below this cell is written yet. Numbers get filled in only after code
runs and produces them.

In [1]:
import sys
from pathlib import Path

# Add the repo root (parent of notebooks/) to sys.path so `src` is importable
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

In [2]:
from src.data import load_hotpotqa

data = load_hotpotqa()
print(f"Loaded {len(data)} examples")
example = data[0]
print("Question:", example["question"])
print("Answer:", example["answer"])
print("Num supporting facts:", len(example["supporting_facts"]["sent_id"]))
print("Num context docs:", len(example["context"]["title"]))

c:\Users\edrin\OneDrive\Desktop\Self Initiated Projects\Agentic-rag-research-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 200 examples
Question: What nationality was Oliver Reed's character in the film Royal Flash?
Answer: Prussian
Num supporting facts: 3
Num context docs: 10


## Data Exploration

Loaded a 200-example slice of HotpotQA (distractor config, validation split,
seed=42) via the Hugging Face `datasets` library.

**Note on dataset source:** the original `hotpot_qa` repo used a Python
loading script, which is no longer supported by recent versions of the
`datasets` library (`datasets>=4.0` dropped script-based loading entirely).
The maintainers have since published a Parquet-format version under
`hotpotqa/hotpot_qa`, which this project uses instead — same data, standard
columnar format, no `trust_remote_code` needed.

**First example, to sanity-check the schema:**
- Question: *"What nationality was Oliver Reed's character in the film Royal Flash?"*
- Answer: `Prussian`
- Supporting facts: 3 sentences
- Context documents: 10

This confirms the "distractor" setup in practice: the model is given 10
context documents, but only a subset of sentences within 2 of them are
actually needed (3 supporting facts) — the rest are plausible-looking
distractors. This is exactly the setup that should punish naive single-shot
RAG if it grabs the wrong subset of documents on the first retrieval pass,
and is the property the baseline-vs-agentic comparison in this project is
designed to test.

**Next step:** build the baseline single-retrieval RAG pipeline first, per
the plan above, and score it before touching the agentic version — so the
"does the agent actually help" question has a real number to compare against
rather than an assumed one.

## Baseline: Single-Shot Retrieval RAG

Before building anything agentic, this builds and smoke-tests the baseline
this project needs to beat: standard retrieve-once-generate-once RAG.

**Retrieval:** each of the 10 context documents per question is embedded
with `all-MiniLM-L6-v2` (local, no API cost) and indexed with FAISS
(exact search — the scale here doesn't warrant an approximate index). The
raw question is embedded once and the top-2 most similar documents are
retrieved. Top-2 matches the known structure of HotpotQA distractor
questions (exactly 2 of the 10 documents contain the supporting facts),
giving the baseline its best realistic shot at grabbing the right documents
in a single pass.

**Generation:** the retrieved documents are handed to Gemini 2.5 Flash in
one call, with an explicit instruction to say "Cannot determine from
context" rather than guess if the retrieved documents don't contain the
answer -- this makes retrieval failures visible in the output rather than
papered over by the model inventing a plausible-sounding wrong answer.

**Why build and test this before the agentic version:** if the baseline
already retrieves the right 2 documents most of the time, that would mean
this particular 200-example slice isn't a strong test of multi-hop
retrieval, and the eventual "agent vs baseline" comparison needs to be
read in that light. The single-example test below is a smoke test only --
full 200-example scoring happens next, once this is confirmed to run
correctly.

In [3]:
from src.baseline import answer_baseline

result = answer_baseline(data[0])
print("Question:", result["question"])
print("Gold answer:", result["gold_answer"])
print("Retrieved docs:", result["retrieved_titles"])
print("Docs actually needed:", result["supporting_titles"])
print("Generated answer:", result["generated_answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1867.02it/s]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Question: What nationality was Oliver Reed's character in the film Royal Flash?
Gold answer: Prussian
Retrieved docs: ['Royal Flash (film)', 'Royal Flash']
Docs actually needed: ['Royal Flash (film)', 'Royal Flash (film)', 'Otto von Bismarck']
Generated answer: Cannot determine from context.


**Smoke test result:**

- Question: *"What nationality was Oliver Reed's character in the film Royal
  Flash?"*
- Gold answer: `Prussian`
- Retrieved: `Royal Flash (film)`, `Royal Flash`
- Actually needed: `Royal Flash (film)` (x2 supporting facts), `Otto von Bismarck`
- Generated answer: `"Cannot determine from context."`

This single example already surfaces the exact failure mode this baseline
is expected to have. The retriever correctly grabbed `Royal Flash (film)`
but pulled a similarly-titled but wrong second document (`Royal Flash`,
likely the novel or character entry) instead of `Otto von Bismarck` --
which has no lexical or obvious semantic overlap with the raw question text.
The connection to Bismarck only becomes findable *after* knowing which
historical figure Oliver Reed's character represents, information the
single-shot retriever never has a chance to use, since it only ever
searches once, on the original question.

Worth noting: the model did not hallucinate a plausible-sounding wrong
answer -- it correctly reported it couldn't determine the answer from what
it was given. That's a deliberate prompt design choice (see the baseline
prompt's explicit instruction), and it matters for scoring: this kind of
honest "I don't know" should be easy to distinguish from a confident wrong
guess when we get to the full 200-example evaluation, since it's a
retrieval failure being correctly reported, not a reasoning failure.

This is one example, not a pattern yet -- Commit 5 runs this across all 200
and scores it properly. But it's a strong early signal that this dataset
slice is a legitimate multi-hop stress test, not a soft one.

In [4]:
from src.run_eval import run_baseline_eval
from src.evaluate import aggregate_results

results = run_baseline_eval(data)  # data = your 200-example slice from Commit 3
summary = aggregate_results(results)
print(summary)

Baseline eval: 100%|██████████| 200/200 [20:19<00:00,  6.10s/it]

{'n_examples': 200, 'mean_em': 0.25, 'mean_f1': 0.310392230212243, 'n_retrieval_success': 63, 'n_retrieval_failure': 137, 'f1_when_retrieval_succeeded': 0.7270421607378129, 'f1_when_retrieval_failed': 0.11879408697785684}


## Baseline Results: Full 200-Example Evaluation

| Metric | Value |
|---|---|
| Exact Match (EM) | 0.250 |
| F1 | 0.310 |
| Retrieval succeeded (both supporting docs retrieved) | 63 / 200 (31.5%) |
| F1 when retrieval succeeded | 0.727 |
| F1 when retrieval failed | 0.119 |

**Headline finding:** answer quality is overwhelmingly determined by whether
retrieval happened to surface the right documents, not by the generation
step. F1 is roughly 6x higher (0.727 vs 0.119) when the single-shot
retriever grabbed both supporting documents versus when it missed at least
one. Since retrieval only succeeds on 31.5% of examples, this baseline's
low overall score (0.310 F1) is substantially a *retrieval* failure, not a
*reasoning* failure -- the model, when given the right evidence, does
reasonably well (0.727 F1); it just isn't given the right evidence most of
the time.

This matches the hypothesis from the background research: a single
embedding search over the raw question has no way to know it needs a
*second*, seemingly unrelated document (like the Oliver Reed/Bismarck case
from the smoke test) until after learning an intermediate fact the first
document reveals. The question text alone under-specifies what to retrieve
for the second hop.

**This is the number the agentic version needs to beat.** Specifically, the
agent's re-retrieval loop should be judged primarily on whether it raises
the 31.5% retrieval-success rate, since that appears to be the leverage
point -- not on generation quality alone, which this baseline shows is
already decent when given adequate context.

In [5]:
import json

with open("../results/baseline_results.jsonl", "r", encoding="utf-8") as f:
    all_results = [json.loads(line) for line in f]

failures = [r for r in all_results if not set(r["supporting_titles"]).issubset(set(r["retrieved_titles"]))]

print(f"{len(failures)} retrieval failures out of {len(all_results)} total\n")

for r in failures[:5]:
    print("Q:", r["question"])
    print("Gold:", r["gold_answer"])
    print("Needed docs:", set(r["supporting_titles"]))
    print("Retrieved docs:", set(r["retrieved_titles"]))
    print("Generated:", r["generated_answer"])
    print("EM:", r["em"], "| F1:", round(r["f1"], 3))
    print("---")

137 retrieval failures out of 200 total

Q: What nationality was Oliver Reed's character in the film Royal Flash?
Gold: Prussian
Needed docs: {'Royal Flash (film)', 'Otto von Bismarck'}
Retrieved docs: {'Royal Flash (film)', 'Royal Flash'}
Generated: Cannot determine from context.
EM: 0 | F1: 0.0
---
Q: Pacific Mozart Ensemble performed which German composer's Der Lindberghflug in 2002?
Gold: Kurt Julian Weill
Needed docs: {'Pacific Mozart Ensemble', 'Kurt Weill'}
Retrieved docs: {'The Flight Across the Ocean', 'Pacific Mozart Ensemble'}
Generated: Kurt Weill
EM: 0 | F1: 0.8
---
Q: What Kentucky county has a population of 60,316 and features the Lake Louisvilla neighborhood?
Gold: Oldham County
Needed docs: {'Oldham County, Kentucky', 'Lake Louisvilla, Louisville'}
Retrieved docs: {'Lake Louisvilla, Louisville', 'Kentucky County, Virginia'}
Generated: Cannot determine from context.
EM: 0 | F1: 0.0
---
Q: Para Hills West, South Australia lies within a city with what estimated population

## Failure Analysis (5-example sample)

**Pattern 1 (Majority): Near miss on similar to adjacent tiles**

**Here are 3 examples of this:**
1. "Royal Flash" was retrieved instead of Otto von Bismarick (the title resembles "Royal Flash (film)")
2. "Kentucky County, Virgina" was retrieved instead of "Oldham County, Kentucky" [term overlap, i.e. ("Kentucky County")]
3. "Para Hills, South Australia" taken instead of County of Salisbury. Suburbs are adjacent, having similar names to the actual neeeded doc

In all three, the retriever wasn't confused by *hops* -- it was pulled
toward a document that superficially resembles the question's subject
(shared words, adjacent geography, similar title), rather than the
document that actually contains the answer. If the agent simply reran
the *same* question as a second search, it would likely make the exact
same mistake again. Fixing this needs a *different* query, not just
another attempt at the same one.

**Pattern 2 (Minority, 1/5): Genuine multi-hop entity gap**

"In what year was the narrator of Blackadder's Christmas Carol born?" --
this needs the character/narrator identified first (Hugh Laurie), then a
second lookup on *that name*. The raw question never mentions "Hugh
Laurie," so no single-shot search over the original text could ever find
the birth-year document. This is the textbook multi-hop case the
background research predicted, and it's the case an agentic re-retrieval
loop is specifically built to solve.

**Caveat: the retrieval-success metric may undercount real successes**

One example (Pacific Mozart Ensemble / Kurt Weill) was labeled a
retrieval failure -- "Kurt Weill" wasn't among the officially-tagged
supporting docs retrieved -- but the model still answered correctly,
because the document that *was* retrieved ("The Flight Across the
Ocean," the English title of the same piece) apparently already
contained the composer's name. This means the 31.5% retrieval-success
figure from the full evaluation is a reasonable proxy, not an exact
count -- some "failures" by that strict metric still produced correct
answers, likely for reasons outside HotpotQA's labeled supporting-fact
titles.

**Implication for agent.py's design:** since the dominant failure mode
(Pattern 1) is near-miss confusion rather than a pure multi-hop gap, the
agent's re-retrieval step needs to extract a specific named entity from
the first retrieval pass and search on *that entity*, not simply resend
a broader version of the original question. A second identical-in-spirit
search would likely repeat the same title-similarity confusion seen
above.


